In [1]:
from pathlib import Path
import json
import numpy as np
import torch

def camera_to_matrix(camera):
    R = torch.tensor(camera["R"], dtype=torch.float32)
    t = torch.tensor(camera["t"], dtype=torch.float32).view(3)
    T = torch.eye(4, dtype=torch.float32)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

def relative_translation_xy(camera1, camera2):
    T1 = camera_to_matrix(camera1)
    T2 = camera_to_matrix(camera2)
    T_rel = T2 @ torch.linalg.inv(T1)
    t = T_rel[:2, 3]
    return t.numpy()

root_dir = Path("dataset")
distances = []

for scene_dir in sorted(root_dir.glob("scene_*")):
    label_file = scene_dir / "labels.json"
    if not label_file.exists():
        continue

    with open(label_file, "r") as f:
        data = json.load(f)

    scene_cam1 = data.get("camera1")
    scene_cam2 = data.get("camera2")

    for p in data.get("pairs", []):
        camera1 = p.get("camera1", scene_cam1)
        camera2 = p.get("camera2", scene_cam2)

        if camera1 is None or camera2 is None:
            continue

        t_xy = relative_translation_xy(camera1, camera2)
        dist = np.linalg.norm(t_xy)
        distances.append(dist)

distances = np.array(distances)

print("antal samples:", len(distances))
print("min:", distances.min())
print("mean:", distances.mean())
print("median:", np.median(distances))
print("p95:", np.percentile(distances, 95))
print("max:", distances.max())

scale_xy = np.percentile(distances, 95) * 1.1
print("föreslagen scale_xy:", scale_xy)

antal samples: 99
min: 0.032945395
mean: 1.2051985
median: 1.1297568
p95: 2.5580573
max: 2.8263626
föreslagen scale_xy: 2.813863
